<a id="top"></a>

# Day 1: What may go into an AI tool

## Learning objectives

<!-- FILL learning objectives: write them in a section headed "Learning objectives" in the day's teaching text, and rebuild -->
**Not written yet.**

## Contents

- [What this week is for, and the case-notes extractor](#what-this-week-is-for-and-the-case-notes-extractor)
  - [The case-notes extractor](#the-case-notes-extractor)
  - [Two models, one task](#two-models-one-task)
  - [The words this chapter uses](#the-words-this-chapter-uses)
- [1. What may go into an AI tool](#1-what-may-go-into-an-ai-tool)
  - [1.1 Keep taxpayer and personal data out of every prompt, log and test fixture](#1-1-keep-taxpayer-and-personal-data-out-of-every-prompt-log-and-test-fixture)
  - [1.2 Work out what a federal rule requires of this task, and who decides](#1-2-work-out-what-a-federal-rule-requires-of-this-task-and-who-decides)
  - [1.3 Tighten the defaults where a repository could reach return information](#1-3-tighten-the-defaults-where-a-repository-could-reach-return-information)
- [The exercise](#the-exercise)
  - [What counts as done](#what-counts-as-done)
  - [The steps](#the-steps)
  - [How it is judged](#how-it-is-judged)

<a id="what-this-week-is-for-and-the-case-notes-extractor"></a>

## What this week is for, and the case-notes extractor

This is the first of ten weeks, and it teaches the way of working that every later week assumes.
Its question is the one you should be able to answer yes to by Friday: **can you make a verified
change to the reference application with an approved coding tool, write a prompt whose results you
can measure, and say what may never go into an AI tool?**

Three things this course expects of a graduate start here: making a change at the speed the tools
allow while being able to point at which part the tool did and which part you judged; bounding what
an agent may touch, and keeping taxpayer and personal data out of every AI tool; and knowing how a
language model produces its output, where it fails, and how a prompt is measured rather than judged
by eye.

Your own agency has already written the obligation down. The IRS's policy on artificial
intelligence, in the Internal Revenue Manual (the IRS's own book of procedures), lists the uses of
generative AI that it prohibits, and one of them is to "Use any outputs from AI systems in the
performance of government business without ensuring direct human review to confirm the veracity of
the information." [[1]](#source-1) The same section puts the inputs on you as
well: "IRS GenAI users assume responsibility for ensuring the accuracy and legality of the data they
upload or input into the AI systems, and the data that is output from them before it may be
shared." [[1]](#source-1) Everything in this chapter is a way of meeting
those two sentences while still getting the speed the tools offer.

[Back to top](#top)

<a id="the-case-notes-extractor"></a>

### The case-notes extractor

One task runs through this chapter, and the week's exercise is to finish it.

![The five days of week 1 left to right in time, each a box with its number above it. Day 1: What may go into an AI tool. Day 2: How a model writes, and how to direct it. Day 3: Forms, tools and a score. Day 4: Git and the coding agent. Day 5: The AI-assisted loop. Under all five, one blue band from the start of day 1 to the end of day 5: The case-notes extractor, a small Python function that reads one note and returns its fields as JSON. An arrow along the bottom: time.](figures/week-one-day-1-of-5.png)

Your lab repository, in the course's hosted environment, holds a set of simulated case notes:
short paragraphs written for this course about synthetic tax returns, of the kind a person working
a compliance case writes as they go. No note describes a real person. Each one says which return it
concerns, which tax year, what looked wrong, what amount is in question if there is one, and what
happens next. Here is one, in the shape the lab uses:

```text
Note 0147. Synthetic filer. Tax year 2023, Form 1040. Schedule C gross receipts are lower
than the information returns on file by 18,400. Sent a letter asking for business records.
Follow up in 30 days.
```

Your job across the week is to build **the case-notes extractor**: a small Python function that
reads one note and returns its fields as JSON that a later program can rely on, and then to find
out how often it is right. For the note above, the result should be this:

```json
{"tax_year": 2023, "form": "1040", "issue": "Schedule C gross receipts below information returns",
 "amount_in_question": 18400, "next_action": "Follow up on records request in 30 days"}
```

A note that names no amount has no amount, and the extractor must say so rather than supply one.

It is a small piece of software, and deliberately a task that pulls on every part of this week:

- **The notes look like the real thing.** In your own work a case note is not simulated. The statute
  that makes tax returns confidential defines return information to include a taxpayer's identity
  and the amounts on a return, and also "whether the taxpayer’s return was, is being, or will be
  examined or subject to other investigation or processing" [[2]](#source-2). A note about a case is
  return information by that definition. So before the first prompt, the question of what may go
  into which tool is already live (part 1).
- **The coding agent will read the repository, run commands, and write tests.** It can also edit
  the tests it is supposed to pass. What it may touch is part 1; how its changes are recorded and
  compared is part 4; how you know its result is right is part 5.
- **The extractor itself calls a language model.** You will direct one model to write code that
  calls another. The second model will sometimes return a confident wrong field, and you cannot see
  how often by reading three outputs (parts 2 and 3).

[Back to top](#top)

<a id="two-models-one-task"></a>

### Two models, one task

Keep the two models in this task apart from the start, because the chapter checks them in
different ways.

| | The coding agent | The model the extractor calls |
|---|---|---|
| What it does | Reads and changes your repository, and runs commands in it | Reads one note and returns its fields |
| Who directs it | You, with a directing prompt | Your code, with a system prompt |
| What it can damage | Files, credentials, history, anything its tools reach | The correctness of every record it produces |
| How you check it | Tests you ran, your own reading of the diff, a done-when | Schema validation, and a score on the evaluation set |
| Where this chapter covers it | Parts 1, 4 and 5 | Parts 2 and 3 |

[Back to top](#top)

<a id="the-words-this-chapter-uses"></a>

### The words this chapter uses

Each of these is defined here once and used without re-definition in later weeks.

A **coding agent** is a model given tools, instructions and a loop, able to read and change a
repository. The **approved coding tools** for this course are GitHub Copilot, Claude Code and Codex
CLI; this chapter names one of them only as an example, never as the only way. Every example of an
agent's own settings in this chapter (its model and its reasoning level, in section 2.3) is Claude
Code's. The sources behind this chapter do not document the equivalent settings in GitHub Copilot or
Codex CLI; find them in that tool's own documentation. The **reference application** is the Python
full-stack application every lab in the course adds to.

What the agent may touch is set by three settings that work together: its **sandbox** (which
files, commands and network it can reach at all), its **approvals** (which actions wait for you to
say yes), and its **tool access** (which tools and credentials it holds). You set all three to the
risk of the repository in front of you.

The **data rules** say what may not go into a prompt or a training set, which approved tool may be
used for which work, and what must be logged. The **data-rules check** is the check you pass before
any lab uses an AI tool; section 1.2 shows its shape.

A **directing prompt** is a task written for the agent as a goal, pointers to the context it needs,
the constraints it must respect, and a **done-when**: the stated condition the result is checked
against, by a test or a run you ran yourself, never by the agent's word.

**Structured output** is model output returned in a format you specified, such as JSON that matches
a schema, rather than free text. **Function calling** is telling a model which functions exist, so
that it answers with a call matching their definition. The **evaluation set** is the course's set of
labelled cases, started this week and grown every week, on which a prompt (and later, whole systems)
is scored.

[Back to top](#top)

<a id="1-what-may-go-into-an-ai-tool"></a>

## 1. What may go into an AI tool

**Before you read on: your lab repository holds only simulated notes. Name three things the coding
agent can reach from it right now that are not in the repository. Then check.**

Nearby files are the obvious answer. The list can also include the credentials in the environment the
agent's shell inherits, every host the network lets it reach, your home directory, and, on the other
side of every prompt, the servers of the company that runs the model. This part sets both halves
of the boundary before the first prompt: **what the agent may touch**, and **what may go into any AI
tool at all**. A boundary set after the work describes what happened; set before, it decides it.

[Back to top](#top)

<a id="1-1-keep-taxpayer-and-personal-data-out-of-every-prompt-log-and-test-fixture"></a>

### 1.1 Keep taxpayer and personal data out of every prompt, log and test fixture

**If you strip the name and the taxpayer identification number out of a real case note, is what is
left safe to paste into a public chat tool?**

The intuitive answer is yes, and it is wrong for two separate reasons.

The first is the law's own test. The law is a statute, section 6103 of the Internal Revenue Code
(26 U.S. Code § 6103, "Confidentiality and disclosure of returns and return information"), and
its definition of return information has a narrow
exception: the term "does not include data in a form which cannot be
associated with, or otherwise identify, directly or indirectly, a particular taxpayer."
[[2]](#source-2) The test is not *no name*. It is *cannot be associated,
directly or indirectly*. A note that gives a tax year, a form, an amount and a follow-up date has
not obviously passed that test because its name line is gone, and deciding that it has is not a call
a developer makes alone.

The second is IRS policy rather than the statute: the policy's list, which does not depend on that
test at all. The IRS policy on AI, Internal Revenue Manual section 10.24.1,
"IRS Policy for Artificial Intelligence (AI) Governance", prohibits putting, into "any public, non-Treasury, or otherwise-unauthorized AI system", a long
list of categories that runs from personally identifiable information and federal taxpayer information
through "statutorily-protected tax information" to "any other non-public record or information that
the Department has identified as restricted in nature" [[1]](#source-1). Its
guidance then closes the gap the intuitive answer relies on: "Employees must be aware that entering
any information, including seemingly innocuous work-related matters, into public or non-Treasury AI
tools (such as publicly accessible chatbots or AI writing assistants) can pose significant risks."
[[1]](#source-1) The reason it gives is about where the text goes: "Publicly-available platforms and services
typically retain, reuse, or resell entered information, potentially resulting in unintended
disclosure of IRS internal strategies, plans, or intentions." [[1]](#source-1)
The UK's National Cyber Security Centre said the same of public chat services in March 2023: "Those queries are
stored and will almost certainly be used for developing the LLM service or model at some point."
[[3]](#source-3) It added: "A question might be sensitive because of data included in the query, or because who is asking the
question (and when)." [[3]](#source-3) A question from an IRS address about one
unusual deduction pattern says something even if no taxpayer is named in it.

The penalty is stated in the same section of the policy: "Undertaking any of the above prohibited
uses may be the basis of disciplinary action, up to and including removal from federal service."
[[1]](#source-1)

**A coding agent is an AI tool too, and it sends more than you type.** A lecture on agentic coding
from MIT's Missing Semester course puts it plainly: "Most AI coding tools in their standard configurations
send a lot of your data to the cloud." [[4]](#source-4) And it is not
only the prompt:

> Sometimes the harness runs locally while LLM inference runs in the cloud, other times even more of
> the software is running in the cloud (and, e.g., the service provider might effectively get a copy
> of your entire repository as well as all interactions you have with the AI tool).
> [[4]](#source-4)

The *harness* there is the program around the model that runs its tools, and *inference* is the
model producing its output. Read the parenthesis against your own repositories. Whatever the agent
can open, it can send, because sending is how it works. So the rule for a coding agent is not *do
not paste taxpayer data*. It is **do not let the agent reach a file that holds it.**

The IRS's security publication for its contractors, Publication 4812, names the tools:
"Contractors are strictly prohibited from disclosing Sensitive but Unclassified (SBU), to include
PII and Federal Tax Information (FTI) information, to publicly available generative AI tools (e.g.,
ChatGPT, Google Bard/Gemini, Anthropic Claude)." [[5]](#source-5)
And it names the act: "The IRS considers sharing data with such an AI or internet tool as an
intentional unauthorized disclosure and data breach." [[5]](#source-5)
That publication binds contractors; your own rule is the policy quoted above. It shows how the IRS
classifies the act, and this course runs in a contractor-hosted environment.

The same publication's minimization rule is a habit for this week: "Many AI models can intake, keep, and reuse data. Under the IRS policies for minimization,
minimize the collection, use, retention, and disclosure of data in AI to what is specifically
relevant and necessary." [[5]](#source-5)

**Prompts are the obvious path. Logs and test fixtures are the two the agent creates for you.** In
the case-notes task, picture the agent's first draft failing schema validation on one note. A
helpful agent adds a line that prints the whole note when validation fails, so you can see why. In
the lab that prints a simulated note. In a repository that held real notes, the same line writes
return information into a log file that nobody decided should hold it. Test fixtures, the sample
inputs a test runs against, are the same: an agent asked for *realistic* test data will produce
realistic-looking data, and a developer who pastes a real note in as a fixture *to be safe* has
committed it.

![A dashed boundary named: Whatever the agent can open, it can send, because sending is how it works. Inside it, top left, in blue, The prompt you type: the obvious path. An arrow labelled type runs to it from You, outside the boundary. Below it, in orange, The two the agent creates for you: A log line, which prints the whole note when validation fails, so you can see why; and A test fixture, the sample inputs a test runs against; a real note pasted in. An arrow labelled creates runs to them from The coding agent, outside the boundary. At the right, in grey, What else it can reach, set by its sandbox, approvals and tool access: nearby files, your home directory, the shell's credentials, every reachable host.](figures/what-a-coding-agent-can-send.png)

Once committed, it is expensive to get back. GitHub's guidance on removing sensitive data: "You
cannot remove sensitive data from other users' clones of your repository" [[6]](#source-6).
For a credential, the same page says the first move is not to rewrite history at all: "as a first
step you need to revoke and/or rotate that secret." [[6]](#source-6) Return
information has no equivalent of rotation, which is why this section is about keeping it out rather
than cleaning it up.

**A written rule is not the same as a closed door.** The Treasury Inspector General for Tax
Administration examined the IRS's AI governance in a report issued on November 12, 2024
("Governance Efforts Should Be Accelerated To Ensure the Safe, Secure, and Trustworthy Development
and Use of Artificial Intelligence"). For October 31, 2023, to January 29, 2024, the traffic logs
showed "531 attempts were made to access ChatGPT, a prohibited generative AI
website." [[7]](#source-7) Those attempts were blocked: "Our review of the
IRS’s traffic logs showed that existing rulesets were working to deny access to generative AI
websites." [[7]](#source-7) But one AI website that had been identified was
never added to the block list, and the logs for the same period "showed 191 successful accesses to
this AI website." [[7]](#source-7) The lesson is structural: the written
rule existed, people still reached the
tool, and what stopped them, where they were stopped, was a rule a machine enforced. Section 1.3
applies that to the coding agent.

![Traffic logs, October 31, 2023, to January 29, 2024. At the left, IRS employees, through the IRS network. At the right, one frame, the written rule, holding two websites: ChatGPT, a prohibited generative AI website; and a specific AI website, that employees should not access. Between them, a column headed the block list, a rule a machine enforced. Top row: a band of 531 attempts toward ChatGPT ends at an orange block, marked with a ban icon; existing rulesets were working to deny access. Bottom row: the block's place is an empty dashed outline, never added to the block list, and a band of 191 successful accesses runs through it to the specific AI website. The two bands are drawn to one scale.](figures/a-rule-a-machine-enforced.png)

**Three risks the agent will not raise for you.** Noticing a risk specific to this data, and saying
so in the pull request, is your part of the work. For the case-notes extractor there are three,
and you name them before you start: the log line on a validation failure; any fixture not written for the course; and any
step that would send a note anywhere other than the model the extractor is approved to call.

[Back to top](#top)

### Exercise: Find and close the three places a note leaks from the extractor

After this you can find where a coding agent's draft lets a case note's text out, to a log, a model that is not approved or a test fixture, and close each path.

**Success criterion:** in the folder, `python3 check.py` prints `Done:`. It runs your `extract` on a failing reply for note 0147 with the network off, and passes when `extract` raises `ExtractionFailed` and tries no host, the log it writes, `logs/extractor.log`, names note 0147 and holds none of its text, every fixture is a course note, unchanged, and the tests pass. Until then it prints `Not yet:` and what is left.

**Where you work:** VS Code in your hosted workspace: `extractor.py` and `tests/fixtures/` in the folder `find-note-leaks`, and the terminal. AI tools: none.

**Start from:** the folder `find-note-leaks` in the practice kit: `cd ~/practice/week-01/day-01/find-note-leaks`, then `python3 check.py`, which prints `Not yet:` and the three places the note's text went. To get it again: `python3 ~/practice/reset.py find-note-leaks`.

#### Steps

1. Find what wrote note 0147's text into `logs/extractor.log`, close that path, and run the check. You should see: `Not yet:` no longer naming the log.
2. Find what sends the note to the host the check names, close that path, and run the check. You should see: `Not yet:` naming only a fixture.
3. Find the fixture that is not a course note and close that path, keeping what its test checks; run `python3 -m pytest -q`. You should see: `3 passed`.
4. Run `python3 check.py`, then read `logs/extractor.log`. You should see: `Done:`, and a line naming note 0147 and why it failed, with none of its text.

**Answer:** `~/practice/week-01/day-01/answers/find-note-leaks/` in the practice kit: the three paths closed in `extractor.py` and `tests/`; there `python3 check.py` prints `Done:`.

The cells below run in your hosted workspace. You edit the files in the editor; the cells run the checks and show the files.

In [ ]:
%cd ~/practice/week-01/day-01/find-note-leaks

The file you edit, `extractor.py`:

In [ ]:
!cat extractor.py

The done check. Run it now, and again after each change: it prints `Not yet:` and what is left, or `Done:`.

In [ ]:
!python3 check.py

The tests:

In [ ]:
!python3 -m pytest -q

What the check left in `logs/extractor.log`:

In [ ]:
!cat logs/extractor.log

#### Afterwards, together

Which of the three paths would the agent never have raised on its own? Which of your searches could a machine run before every commit?

<a id="1-2-work-out-what-a-federal-rule-requires-of-this-task-and-who-decides"></a>

### 1.2 Work out what a federal rule requires of this task, and who decides

**Pretend the case-notes extractor will one day run on real notes. Which of its inputs are
protected, which step crosses a boundary, and what does that require you to build or not build,
before anyone writes a line of it?**

Knowing that a rule exists is the easy half. The other half is working out what it requires of one
task. There are five moves, in order.

1. **Classify the protected surface.** Name the fields, the files and the steps the rule reaches.
   For the extractor, every field is inside the statute's definition, which covers a taxpayer's
   identity, the amounts on a return, and whether a return is being examined
   [[2]](#source-2). The step that crosses a boundary is the one where a
   note's text leaves your system for the model the extractor calls. The steps that create new
   copies are the log line and the test fixture from section 1.1.
2. **Name what the classification triggers, in delivery terms.** For the extractor: only an
   approved model; only the note text the extraction needs, and nothing else from the case, which
   is the minimization rule; an audit trail of every note processed; no real note in any fixture;
   and logs that hold note text treated as holding return information.
3. **State the forbidden actions.** Sending a note to "any public, non-Treasury, or
   otherwise-unauthorized AI system" [[1]](#source-1). Using the extractor's
   output as a decision: the policy's guidance says "Do not use GenAI to make binding determinations
   or actions on taxpayer rights without proper oversight" [[1]](#source-1),
   so the extractor's fields go to a person, never straight into an action on a case.
4. **Write it as a requirement someone can act on.** Not *be careful with data*, but a line a
   reviewer can check: *the extractor calls only the approved model endpoint named in the
   configuration; it logs the note identifier and never the note text; its fixtures are the
   course's simulated notes and are marked as such.*
5. **Route it to the owner.** Approval of the use goes to the accountable official the policy
   names (section 1.3 quotes the rule). Questions about using data in generative AI, including
   synthetic and simulated data, go to the privacy contact the policy names in its considerations
   [[1]](#source-1). You do not resolve either one by writing code.

**The data-rules check.** Before any lab in this course uses an AI tool, you pass a short check, and
this week's lab starts with it. The check is this course's own. Its shape is a set of
combinations, each an item of information and a tool, that you classify as permitted or not and
give the reason for, citing the rule. Six of them, for this week:

| Information | Tool | Permitted? | Why |
|---|---|---|---|
| A simulated note from the lab repository | The approved coding tool, in the course's hosted environment | Yes | The notes are written for the course and describe no real person |
| A real case note from your own work | A public chat tool | No | Prohibited input into a public or unauthorized AI system |
| The same real note with the name and identification number removed | A public chat tool | No | Removing a name does not show the note can no longer be linked to a taxpayer, and the prohibition also covers any other non-public record |
| Your own code, with no taxpayer data in it | A public chat tool | No | Only approved tools may be used, and the policy warns that even harmless-looking work matters carry risk in public tools |
| A real case note | An approved tool, for work that tool has not been approved for | Not yours to decide | Approval of each use rests with an accountable official |
| The agent's session log from the lab | Attached to your pull request as evidence | Yes | It holds only simulated data, and keeping it is part of the record |

Every *No* rests on a rule quoted in this section or in sections 1.1 and 1.3, and the row that is not yours
to decide is the one this week trains you to notice.

[Back to top](#top)

### Exercise: Write the extractor's data requirements, then pass the data-rules check

After this you can work out what a federal data rule requires of the case-notes extractor, write it as requirements a reviewer can check, route each open question to the role that answers it, and pass the data-rules check before any AI tool touches the repository.

**Success criterion:** from `lab`, `python3 ../check.py` prints a line starting `Done:`: the check's six answers are the course's, each with a reason, and each reason for an answer other than Yes ends with the section its rule is in; the TODO cells of `docs/requirements.md` are filled; and both files are committed, with no commit an AI tool made before them. Until then it prints `Not yet:` and what is left. The check tests that each reason and requirement is there and in that shape, not how good it is. Nothing is handed in; the class talks it through when the exercise ends.

**Where you work:** VS Code in your hosted workspace: the files `docs/data-rules-check.md` and `docs/requirements.md` in the repository `lab` in the folder `data-rules-check`, and the terminal (Terminal > New Terminal), inside `lab`. AI tools: none; no AI tool is opened until the check is committed, and this exercise opens none.

**Start from:** the folder `data-rules-check` in the practice kit: `python3 ~/practice/reset.py data-rules-check`, then `cd ~/practice/week-01/day-01/data-rules-check/lab`. The reset command builds the repository `lab` and prints `data-rules-check is back as it was supplied.` To get it again: the same reset command, which also removes your commits. Git commits under the name and email you gave it in the practice environment's set-up.

#### Steps

1. In the terminal, inside `lab`, see where you start and what is left: `git log --oneline`, `git status --short`, then `python3 ../check.py`. You should see: one commit by the course team, `?? docs/data-rules-check.md` and `?? docs/requirements.md` (two files not committed), and `Not yet:` naming the empty rows, the TODO cells and the two files.
2. Take the check: in the Explorer on the left, open `practice`, then `week-01`, `day-01`, `data-rules-check`, `lab` and `docs`, then `data-rules-check.md`. In each of the six rows, write in Permitted? Yes, No, or Not mine to decide where the answer is someone else's to give, and the reason in Why; in each row that is not Yes, end the reason with the section of the chapter whose rule it rests on, written (section 1.1), (section 1.2) or (section 1.3). Save it (File > Save), then run `python3 ../check.py`. You should see: `Not yet:` naming no row of `docs/data-rules-check.md`; a row it names is not answered as the course's check answers it, and the check tells you to compare the table in section 1.2.
3. Write the requirements and route the questions: open `requirements.md`, in the same `docs` folder. In its table of requirements, the first column of each of the three TODO rows already says what that requirement covers. In each, replace the TODO in Requirement a reviewer can check with a line a reviewer can check from the repository alone, and the TODO in Where they check it with the file or folder where they check it, in backticks, such as `extractor.py`; a file the lab adds later in the week, as it adds `extractor.py`, may be named. Then, in the table of open questions, write in Who answers it the role the chapter's fifth move names for each question, in the chapter's words. Save, then run `python3 ../check.py`. You should see: `Not yet:` naming only the two files as not committed yet.
4. Commit both, before any AI tool touches the repository: `git add docs/data-rules-check.md docs/requirements.md`, then `git commit -m "Record the data-rules check, passed, and the extractor's requirements"`, then `python3 ../check.py` and `git log --oneline`. You should see: `2 files changed`, a line starting `Done:`, and your commit above the course team's.

**Answer:** `~/practice/week-01/day-01/answers/data-rules-check/` in the practice kit: `data-rules-check.md` and `requirements.md` filled in, and `commands.sh`, the commands in order, each with a note; run from `lab` after the reset command, it ends with `Done:`.

#### Afterwards, together

Which requirement did you rewrite most before a reviewer could check it? Why does the course train you to notice the row that is not yours to decide?

<a id="1-3-tighten-the-defaults-where-a-repository-could-reach-return-information"></a>

### 1.3 Tighten the defaults where a repository could reach return information

**The lab's settings, in the first column of the table below, are enough for simulated notes. What
would you change before pointing the same agent at a repository that could reach real return
information, and who would have to agree?**

Start from the statute's default, because everything else is an exception to it: "Returns and
return information shall be confidential, and except as authorized by this title" no officer or
employee of the United States may disclose them [[2]](#source-2). The
exceptions quoted here are each tied to a purpose and to need. Inside Treasury:

> Returns and return information shall, without written request, be open to inspection by or
> disclosure to officers and employees of the Department of the Treasury whose official duties
> require such inspection or disclosure for tax administration purposes.
> [[2]](#source-2)

And to any person, under the Secretary's regulations, disclosure is allowed only "to the
extent necessary in connection with the processing, storage,
transmission, and reproduction of such returns and return information, the programming,
maintenance, repair, testing, and procurement of equipment, and the providing of other services, for
purposes of tax administration." [[2]](#source-2) Read the shape rather
than the clauses: a named purpose, a named need, and nothing by default. A model provider your
coding agent sends a repository to is not on that list because a developer found it useful.

The policy turns that into three rules a developer meets directly. Only approved tools: "IRS GenAI
users must always ensure the use of only Treasury- or IRS-approved GenAI products and services"
[[1]](#source-1). Each use approved by someone accountable: "IRS uses of GenAI
must be approved by an accountable agency official and align with IRS mission objectives and comply
with applicable legal, privacy, and civil liberties requirements." [[1]](#source-1)
And a record: "AI systems must include audit trails for all access to and processing of SBU/CUI
including PII and FTI." [[1]](#source-1) (SBU is sensitive but unclassified
information; CUI is controlled unclassified information; PII is personally identifiable
information; FTI is federal taxpayer information, as the policy expands it.) The contractor rule says the same about who decides:
"Contractors and subcontractors must not employ AI technologies to support the IRS contract without
consultation with IRS Cybersecurity and approval from the Contracting Officer."
[[5]](#source-5)

**Why a regulated repository needs more than good settings.** In the lab, a setting that usually
holds is good enough, because the worst case is a broken lab. Where return information is
reachable, *usually* is the wrong standard, because a disclosure cannot be undone. And the defences
against an instruction smuggled into the agent's input, which the security standards body OWASP
calls *prompt injection*, are by its own account not complete: "Given the stochastic influence at the heart of the way
models work, it is unclear if there are fool-proof methods of prevention for prompt injection."
[[8]](#source-8) So in a regulated repository you remove the combinations that
would let a bad choice do damage. The one that matters most is an agent that can both read protected
data and reach the network, because together those are a path out.

**The boundary this course proposes for an agent in a regulated repository.** The contractor
publication sets the floor for any code an AI tool writes: "Contractors using AI-assisted software
development tools to generate, modify, or recommend source code must ensure that AI-generated code
is subject to the organization's approved SDLC." [[5]](#source-5)
(SDLC is the software development life cycle.) That publication binds contractors; with the approval
and audit-trail rules quoted above, it supports three rows below: which tool, what the agent may
change, and the record. For the network it sets a floor as well: where AI systems are used in
support of an IRS contract, model access endpoints must be "restricted to authorized personnel and
approved systems" [[5]](#source-5). Neither publication says how
the agent itself is sandboxed or approved action by action while it works. Those rows, and how far
the network row goes beyond that floor, are this course's proposal, offered for the people who decide:

| Setting | Lab default | Regulated repository |
|---|---|---|
| Which tool | Any of the approved coding tools | Only a tool approved for that system and that work, with the approval recorded |
| Where it runs | The lab sandbox | An isolated environment with no route to production data the task does not need |
| Network | Package installs from the lockfile | None beyond what the task needs, and never alongside reachable protected data |
| What it may change | Files in the lab repository | A branch. It opens a pull request; it never merges |
| Approvals | Every command that is not a read, a test run or an edit in the repository | Every command that is not a read or a test run |
| Record | The session log, kept as evidence | The session log, plus the audit trail the system itself must keep |

**What you do not decide.** Proposing the boundary is yours. Deciding that a use is lawful and
approved is not: the policy puts that with an accountable agency official, and the contractor rule
with IRS Cybersecurity and the Contracting Officer. Your job is to make the proposal concrete enough
for them to say yes or no, which is what the five moves in section 1.2 are for.

[Back to top](#top)

### Exercise: Probe the lab's agent boundary, then propose the regulated one

After this you can check that a coding agent's configuration does what it says, and propose, row by row, the tighter boundary for a repository that could reach return information, with the rule each row rests on and the role who decides it.

**Success criterion:** from `lab`, `python3 ../check.py` prints a line starting `Done:`: the first line of `docs/boundary-proposal.md` says it is a proposal and not an approval; every row names the rule it rests on, as a phrase quoted in double quotes or the words course proposal, and the role who decides; and the combination the proposal removes is written. Until then it prints `Not yet:` and what is left. The check tests that each is there and in that shape, not how good it is. Nothing is handed in; the class talks it through when the exercise ends.

**Where you work:** VS Code in your hosted workspace: the file `docs/boundary-proposal.md` in the repository `lab` in the folder `agent-boundary`, and the terminal (Terminal > New Terminal), inside `lab`. AI tools: Claude Code, the coding tool, started in the terminal for one step, to test what its configuration allows.

**Start from:** the folder `agent-boundary` in the practice kit: `python3 ~/practice/reset.py agent-boundary`, then `cd ~/practice/week-01/day-01/agent-boundary/lab`. The reset command builds the repository `lab` and prints `agent-boundary is back as it was supplied.` To get it again: the same reset command.

#### Steps

1. In the terminal, inside `lab`, read the history and the configuration: `git log --oneline`, then `cat .claude/settings.json`. You should see: three commits by the course team, the configuration before the data-rules check; and in the configuration `"defaultMode": "acceptEdits"`, `Edit(/**)` and the test and Git reading commands under `allow`, and `WebFetch`, `WebSearch`, `curl` and `wget` under `deny`, which keep the agent off the web.
2. Test the configuration before you trust it: still inside `lab`, start the coding tool with `claude`; if it asks whether you trust the files in this folder, answer yes, which lets the folder's settings apply; if it asks you to sign in, stop and tell your trainer. Type `Run exactly this command and show me its first line: curl -sI https://example.com` and press Enter. You should see: Claude Code tries the command and reports that permission to use it "has been denied", and no page appears. If it asks to fetch the page some other way, choose the option that starts with No, or press Esc. If it shows the page's first line instead, which starts `HTTP`, the configuration did not keep the agent off the web: tell your trainer. Type `/exit` to leave.
3. Write the proposal: in the Explorer on the left, open `practice`, then `week-01`, `day-01`, `agent-boundary`, `lab` and `docs`, then `boundary-proposal.md`. Replace its first line, which says TODO, with a sentence that says it is a proposal and not an approval, in those words. In each row, write in Rule the rule it rests on, as a phrase from section 1.3 in double quotes, or the words course proposal where the chapter says no rule sets it, and in Who decides the role, in the chapter's words; last, on the file's last line, replace the TODO after `The combination this proposal removes, and why:` with that combination and why, in your words or the chapter's. Save it (File > Save), then run `python3 ../check.py`. You should see: a line starting `Done:`; until then, `Not yet:` and the rows still to fill.

**Answer:** `~/practice/week-01/day-01/answers/agent-boundary/` in the practice kit: `boundary-proposal.md` filled in, and `commands.sh`, which copies it into `docs/` and runs the done check, ending `Done:`; its comments give the probe's expected result.

#### Afterwards, together

Which row needed a role you could not name? What did the probe show that reading the configuration did not?

<a id="the-exercise"></a>

## The exercise

This is the week's lab: the case-notes extractor, built from an empty function to a pull request
with a recorded score, in steps that each end in a saved, working state, so that if your duties call
you away you can stop at the end of any step and pick up at the next.

[Back to top](#top)

<a id="what-counts-as-done"></a>

### What counts as done

State this before you start, and check it at the end. You are done when the lab repository holds
all four of these:

1. **The agent's configuration**, committed before your first prompt: its sandbox, approvals and
   tool access for this repository, from the lab column of the table in section 1.3.
2. **The data-rules check, passed**, recorded before any AI tool touches the repository, with a
   reason for each combination.
3. **A pull request for the extractor** whose description states its done-when and cites the test
   run you ran yourself that shows the done-when met: the output of the command, not a sentence
   saying it passed.
4. **The evaluation set, the prompt, its output schema and its recorded score**, with a short note
   naming one failure the set caught and its cause.

The agent's session log goes with the pull request as added evidence. It is never the only
evidence for any of the four: each one must be checkable from the repository itself.

[Back to top](#top)

<a id="the-steps"></a>

### The steps

Each step ends in a commit. If a step's checks do not pass, it has not ended.

1. **Configure, then check.** Write the agent's configuration for the lab repository and commit it.
   Then do the data-rules check and commit the record. Saved state: a repository that has not yet
   been touched by an AI tool, with a configuration and a passed check in its history.
2. **Run 1: the schema and its tests, with no model call.** Direct the agent to write the output
   schema for the five fields (with amount_in_question allowed to be null) and tests that validate
   a few hand-written outputs against it, including one that should fail. Run the tests yourself.
   Read the tests, then add them to the deny rule before run 2. Saved state: a schema and passing
   tests, and nothing that calls a model.
3. **Run 2: the extractor.** Direct the agent to write the function that sends one note to the model
   with your system prompt and structured output, and validates the reply against the schema. Its
   tests use a recorded model reply, so they run without calling the model. Read the diff before you
   accept it. Saved state: an extractor whose tests pass on recorded replies.
4. **Run 3: the evaluation set and a score.** Label at least 30 of the lab's notes by hand with the
   fields they should produce, including every note in the repository with no amount. Have the agent
   write a scorer that reports, per field, how many extracted values were correct and how many
   expected values were found. Run it and commit the set, the scorer and the score. Saved state: a
   first recorded score.
5. **Revise once, on a named failure.** Pick the failure that cost the most, write down its cause,
   change the prompt (not the scorer and not the labels), and run the whole set again. Commit the
   new prompt, the new score and the note. A lower score is a valid result if you can say why.
   Saved state: two recorded scores and the note between them.
6. **Open the pull request.** Its description states the done-when, cites the command output that
   shows it met, reports both scores, and names the model and reasoning level each run used, with the
   reason (section 2.3). Attach the session log. Saved state: the whole week, reviewable by someone
   who was not there.

![A history read top to bottom: the lab's six steps, one row each, each ending in a commit drawn as a blue dot, with the commit's saved state beside it, under the heading Saved state and the note Each step ends in a commit. Arrows run from each commit down to the next step. First row, inside an orange band labelled before any AI tool touches the repository: Configure, then check; a configuration and a passed check in its history. An orange line closes the band, and the arrow out of the first commit crosses it. Below the line: Run 1: the schema and its tests; a schema and passing tests, and nothing that calls a model. Run 2: the extractor; an extractor whose tests pass on recorded replies. Run 3: the evaluation set and a score; a first recorded score. Revise once, on a named failure; two recorded scores and the note between them. Open the pull request; the whole week, reviewable by someone who was not there.](figures/the-lab-in-six-steps.png)

[Back to top](#top)

<a id="how-it-is-judged"></a>

### How it is judged

You pass when someone who did not watch you work can confirm all four items from the repository and
the pull request alone. Three things fail it, whatever else is good: a configuration or a data-rules
check committed after the first AI-tool commit; a done-when whose only evidence is the agent's
statement that the tests pass; and a score with no evaluation set behind it, or a prompt revised
without the set being run again.

[Back to top](#top)

## Sources

1. <a id="source-1"></a>[10.24.1 IRS Policy for Artificial Intelligence (AI) Governance](https://www.irs.gov/irm/part10/irm_10-024-001r), last updated 2026-08-10.
2. <a id="source-2"></a>[26 U.S. Code § 6103 - Confidentiality and disclosure of returns and return information](https://www.law.cornell.edu/uscode/text/26/6103), last updated 2025-01-04.
3. <a id="source-3"></a>[ChatGPT and large language models: what's the risk?](https://www.ncsc.gov.uk/blog-post/chatgpt-and-large-language-models-whats-the-risk), last updated 2025-03-12.
4. <a id="source-4"></a>[Agentic Coding](https://missing.csail.mit.edu/2026/agentic-coding/), last updated 2026-02-24.
5. <a id="source-5"></a>[IRS Publication 4812, Contractor Security & Privacy Controls: Handling and Protecting IRS Information and Information Systems (Rev. 9-2026)](https://www.irs.gov/pub/irs-pdf/p4812.pdf), last updated 2026-09-28.
6. <a id="source-6"></a>[Removing sensitive data from a repository](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/removing-sensitive-data-from-a-repository).
7. <a id="source-7"></a>[Governance Efforts Should Be Accelerated To Ensure the Safe, Secure, and Trustworthy Development and Use of Artificial Intelligence (TIGTA Report 2025-IE-R003)](https://www.tigta.gov/sites/default/files/reports/2025-08/2025ier003fr_3.pdf), last updated 2024-11-12.
8. <a id="source-8"></a>[LLM01:2025 Prompt Injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/), last updated 2025-04-17.

[Back to top](#top)